In [32]:
from sklearn.ensemble import RandomForestClassifier

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score
)

pd.set_option("display.max_columns", None)

In [3]:
orders = pd.read_csv(
    "../02_data/raw/olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    "../02_data/raw/olist_order_items_dataset.csv"
)

customers = pd.read_csv(
    "../02_data/raw/olist_customers_dataset.csv"
)

print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Customers:", customers.shape)

Orders: (99441, 8)
Order items: (112650, 7)
Customers: (99441, 5)


In [4]:
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

In [5]:
model_orders = orders[
    orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
].copy()

In [6]:
model_orders["delivery_delay_days"] = (
    model_orders["order_delivered_customer_date"]
    - model_orders["order_estimated_delivery_date"]
).dt.days

model_orders["is_late"] = (
    model_orders["delivery_delay_days"] > 0
).astype(int)

In [7]:
print(model_orders["is_late"].value_counts())
print()
print(model_orders["is_late"].value_counts(normalize=True) * 100)

is_late
0    89941
1     6535
Name: count, dtype: int64

is_late
0    93.226295
1     6.773705
Name: proportion, dtype: float64


In [8]:
order_features = (
    order_items
    .groupby("order_id")
    .agg(
        total_order_value=("price", "sum"),
        total_freight=("freight_value", "sum"),
        number_of_items=("order_item_id", "count"),
        number_of_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

order_features.head()

,order_id,total_order_value,total_freight,number_of_items,number_of_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1


In [9]:
print(order_features.shape)
print(order_features.isna().sum())

(98666, 5)
order_id             0
total_order_value    0
total_freight        0
number_of_items      0
number_of_sellers    0
dtype: int64


In [10]:
customer_features = customers[
    ["customer_id", "customer_city", "customer_state"]
].copy()

model_orders = model_orders.merge(
    customer_features,
    on="customer_id",
    how="left"
)

print(model_orders.shape)
print(model_orders[["customer_city", "customer_state"]].isna().sum())

(96476, 12)
customer_city     0
customer_state    0
dtype: int64


In [11]:
model_data = model_orders.merge(
    order_features,
    on="order_id",
    how="inner"
)

print(model_data.shape)

(96476, 16)


In [12]:
sellers = pd.read_csv(
    "../02_data/raw/olist_sellers_dataset.csv"
)

print("Sellers:", sellers.shape)

Sellers: (3095, 4)


In [13]:
seller_features = sellers[
    ["seller_id", "seller_city", "seller_state"]
].copy()

In [14]:
items_with_sellers = order_items.merge(
    seller_features,
    on="seller_id",
    how="left"
)

In [15]:
seller_order_features = (
    items_with_sellers
    .groupby("order_id")
    .agg(
        seller_city=("seller_city", "first"),
        seller_state=("seller_state", "first"),
        seller_count=("seller_id", "nunique")
    )
    .reset_index()
)

In [16]:
model_data = model_data.merge(
    seller_order_features,
    on="order_id",
    how="left"
)

print(model_data.shape)
print(
    model_data[
        ["seller_city", "seller_state", "seller_count"]
    ].isna().sum()
)

(96476, 19)
seller_city     0
seller_state    0
seller_count    0
dtype: int64


In [17]:
model_data["same_state"] = (
    model_data["customer_state"] == model_data["seller_state"]
).astype(int)

In [18]:
model_data["purchase_date"] = pd.to_datetime(
    model_data["order_purchase_timestamp"]
)

model_data["purchase_month"] = (
    model_data["purchase_date"].dt.month
)

In [19]:
model_data["estimated_delivery_days"] = (
    pd.to_datetime(model_data["order_estimated_delivery_date"])
    - model_data["purchase_date"].dt.normalize()
).dt.days

In [20]:
print(
    model_data[
        [
            "same_state",
            "purchase_month",
            "estimated_delivery_days"
        ]
    ].describe()
)

         same_state  purchase_month  estimated_delivery_days
count  96476.000000    96476.000000             96476.000000
mean       0.359706        6.031210                24.374217
std        0.479917        3.228532                 8.760837
min        0.000000        1.000000                 3.000000
25%        0.000000        3.000000                19.000000
50%        0.000000        6.000000                24.000000
75%        1.000000        8.000000                29.000000
max        1.000000       12.000000               156.000000


In [21]:
print(
    model_data["same_state"]
    .value_counts(normalize=True)
    .mul(100)
)

same_state
0    64.029396
1    35.970604
Name: proportion, dtype: float64


In [22]:
print(
    model_data[
        [
            "purchase_month",
            "estimated_delivery_days"
        ]
    ].describe()
)

       purchase_month  estimated_delivery_days
count    96476.000000             96476.000000
mean         6.031210                24.374217
std          3.228532                 8.760837
min          1.000000                 3.000000
25%          3.000000                19.000000
50%          6.000000                24.000000
75%          8.000000                29.000000
max         12.000000               156.000000


In [23]:
print(
    model_data[
        [
            "order_purchase_timestamp",
            "purchase_date",
            "purchase_month",
            "estimated_delivery_days"
        ]
    ].head()
)

  order_purchase_timestamp       purchase_date  purchase_month  \
0      2017-10-02 10:56:33 2017-10-02 10:56:33              10   
1      2018-07-24 20:41:37 2018-07-24 20:41:37               7   
2      2018-08-08 08:38:49 2018-08-08 08:38:49               8   
3      2017-11-18 19:28:06 2017-11-18 19:28:06              11   
4      2018-02-13 21:18:39 2018-02-13 21:18:39               2   

   estimated_delivery_days  
0                       16  
1                       20  
2                       27  
3                       27  
4                       13  


In [24]:
features = [
    "total_order_value",
    "total_freight",
    "number_of_items",
    "number_of_sellers",
    "same_state",
    "purchase_month",
    "estimated_delivery_days",
    "customer_state",
    "seller_state"
]

X = model_data[features].copy()
y = model_data["is_late"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nMissing values:")
print(X.isna().sum().sum())

X shape: (96476, 9)
y shape: (96476,)

Missing values:
0


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining late rate:")
print(y_train.mean() * 100)

print("\nTest late rate:")
print(y_test.mean() * 100)

Training set: (77180, 9)
Test set: (19296, 9)

Training late rate:
6.773775589530967

Test late rate:
6.773424543946932


In [26]:
numeric_features = [
    "total_order_value",
    "total_freight",
    "number_of_items",
    "number_of_sellers",
    "same_state",
    "purchase_month",
    "estimated_delivery_days"
]

categorical_features = [
    "customer_state",
    "seller_state"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [27]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [28]:
logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['total_order_value',
                                                   'total_freight',
                                                   'number_of_items',
                                                   'number_of_sellers',
                                                   'same_state',
                                                   'purchase_month',
                                                   'estimated_delivery_days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['customer_state',
                                                   'seller_state'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [29]:
y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:, 1]

In [30]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6662002487562189
ROC-AUC: 0.7062475227677817

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.67      0.79     17989
           1       0.12      0.64      0.21      1307

    accuracy                           0.67     19296
   macro avg       0.54      0.65      0.50     19296
weighted avg       0.91      0.67      0.75     19296



In [31]:
feature_names = logistic_model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = logistic_model.named_steps[
    "classifier"
].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

feature_importance["abs_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "abs_coefficient",
    ascending=False
)

feature_importance.head(20)

,feature,coefficient,abs_coefficient
40,cat__seller_state_MA,1.417548,1.417548
8,cat__customer_state_AL,1.258096,1.258096
24,cat__customer_state_PR,-1.019540,1.019540
17,cat__customer_state_MG,-0.944426,0.944426
13,cat__customer_state_DF,-0.829154,0.829154
27,cat__customer_state_RO,-0.827613,0.827613
32,cat__customer_state_SP,-0.805996,0.805996
20,cat__customer_state_PA,0.781121,0.781121
16,cat__customer_state_MA,0.721584,0.721584
31,cat__customer_state_SE,0.700998,0.700998


In [33]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [34]:
random_forest.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['total_order_value',
                                                   'total_freight',
                                                   'number_of_items',
                                                   'number_of_sellers',
                                                   'same_state',
                                                   'purchase_month',
                                                   'estimated_delivery_days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['customer_state',
                                                   'seller_state'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=12,
                                        min_samples_leaf=5, n_estimators=200,
                                        n_jobs=-1, random_state=42))])

In [35]:
rf_pred = random_forest.predict(X_test)
rf_prob = random_forest.predict_proba(X_test)[:, 1]

In [36]:
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Accuracy: 0.7810945273631841
ROC-AUC: 0.7735724369176895

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.79      0.87     17989
           1       0.18      0.63      0.28      1307

    accuracy                           0.78     19296
   macro avg       0.57      0.71      0.58     19296
weighted avg       0.91      0.78      0.83     19296



In [37]:
rf_feature_names = random_forest.named_steps[
    "preprocessor"
].get_feature_names_out()

rf_importances = random_forest.named_steps[
    "classifier"
].feature_importances_

rf_importance = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_importances
}).sort_values(
    "importance",
    ascending=False
)

rf_importance.head(20)

,feature,importance
5,num__purchase_month,0.277047
6,num__estimated_delivery_days,0.186285
1,num__total_freight,0.114287
0,num__total_order_value,0.081566
32,cat__customer_state_SP,0.063976
25,cat__customer_state_RJ,0.048018
4,num__same_state,0.040140
17,cat__customer_state_MG,0.029586
55,cat__seller_state_SP,0.019294
2,num__number_of_items,0.014112


In [38]:
rf_importance.head(15)

,feature,importance
5,num__purchase_month,0.277047
6,num__estimated_delivery_days,0.186285
1,num__total_freight,0.114287
0,num__total_order_value,0.081566
32,cat__customer_state_SP,0.063976
25,cat__customer_state_RJ,0.048018
4,num__same_state,0.040140
17,cat__customer_state_MG,0.029586
55,cat__seller_state_SP,0.019294
2,num__number_of_items,0.014112


In [39]:
rf_prob = random_forest.predict_proba(X_test)[:, 1]

rf_prob[:10]

array([0.59348954, 0.63817622, 0.3752798 , 0.51292765, 0.58112653,
       0.29941002, 0.43819224, 0.41121733, 0.60161916, 0.24569042])

In [40]:
from sklearn.metrics import precision_score, recall_score

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]

threshold_results = []

for threshold in thresholds:
    predictions = (rf_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "flagged_orders": predictions.sum(),
        "flagged_rate": predictions.mean() * 100,
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions)
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,threshold,flagged_orders,flagged_rate,precision,recall
0,0.2,18741,97.123756,0.069313,0.993879
1,0.3,15131,78.415216,0.081951,0.948738
2,0.4,8853,45.879975,0.117926,0.798776
3,0.5,4561,23.637023,0.180224,0.628921
4,0.6,2038,10.561774,0.256133,0.399388


In [41]:
rf_pred = (rf_prob >= 0.50).astype(int)

model_results = X_test.copy()

model_results["actual_late"] = y_test.values
model_results["late_probability"] = rf_prob
model_results["predicted_late"] = rf_pred

model_results.head()

,total_order_value,total_freight,number_of_items,number_of_sellers,same_state,purchase_month,estimated_delivery_days,customer_state,seller_state,actual_late,late_probability,predicted_late
824,299.00,16.84,1,1,0,11,32,RS,SP,0,0.593490,1
62957,69.90,19.73,1,1,0,2,31,BA,SP,0,0.638176,1
93587,119.90,12.04,1,1,1,11,23,SP,SP,1,0.375280,0
41618,25.00,15.28,1,1,0,8,17,RJ,SP,0,0.512928,1
26640,14.99,15.10,1,1,0,3,24,PR,SP,0,0.581127,1


In [43]:
model_results.to_csv(
    "../07_powerbi/delivery_risk_predictions.csv",
    index=False
)

print("Saved successfully.")
print(model_results.shape)

Saved successfully.
(19296, 12)
